In [0]:
#%pip install great-expectations==0.18.22
%pip install great-expectations

In [0]:
%restart_python

In [0]:
import great_expectations as gx
from great_expectations.checkpoint import Checkpoint
from great_expectations.data_context.types.base import DataContextConfig

data_context_config = DataContextConfig(
    plugins_directory=None,
    config_variables_file_path=None,
    stores={
        "expectations_store": {
            "class_name": "ExpectationsStore",
            "store_backend": {
                "class_name": "TupleFilesystemStoreBackend",
                "base_directory": r"/Workspace/Users/snehashrivastav@gmail.com/databricks-repo-fact/gx/expectations",
            }
        },
        "checkpoint_store": {
            "class_name": "CheckpointStore",
            "store_backend": {
                "class_name": "TupleFilesystemStoreBackend",
                "base_directory": r"/Workspace/Users/snehashrivastav@gmail.com/databricks-repo-fact/gx/checkpoints/",
            }
        },
        "validations_store": {
            "class_name": "ValidationsStore",
            "store_backend": {
                "class_name": "TupleFilesystemStoreBackend",
                "base_directory": r"/Workspace/Users/snehashrivastav@gmail.com/databricks-repo-fact/gx/validations/",
            }
        },
        "evaluation_parameter_store": {
            "class_name": "EvaluationParameterStore",
        },
    },
    expectations_store_name="expectations_store",
    validations_store_name="validations_store",
    checkpoint_store_name="checkpoint_store",
    evaluation_parameter_store_name="evaluation_parameter_store",
    data_docs_sites={
        "local_site": {
            "class_name": "SiteBuilder",
            "store_backend": {
                "class_name": "TupleFilesystemStoreBackend",
                "base_directory": r"/Workspace/Users/snehashrivastav@gmail.com/databricks-repo-fact/gx/data_docs/",
            },
            "site_index_builder": {
                "class_name": "DefaultSiteIndexBuilder",
                "show_cta_footer": False
            }
        }
    },
    anonymous_usage_statistics={
        "enabled": True
    }

)

context = gx.get_context(project_config=data_context_config)
display(context)


In [0]:
df = spark.read.table("onedevcatalog.bronze_one.customer_dim")

In [0]:
df.createOrReplaceTempView("temp_customer_dim")

In [0]:
datasource_name = "cust_trans_datasource"
dataframe_datasource = context.sources.add_or_update_spark(name=datasource_name)


In [0]:
dataasset_name = "cust_trans_dataasset"
dataframe_asset = dataframe_datasource.add_dataframe_asset( #add_sql_asset( 
    name = dataasset_name,
    dataframe=df #query="SELECT * FROM temp_customer_dim" 
)

batch_request = dataframe_asset.build_batch_request()

display(dataframe_asset)


In [0]:
expectation_suite_name = "cust_dim_suite"

context.add_or_update_expectation_suite(expectation_suite_name=expectation_suite_name)
validator = context.get_validator(
    #dataframe=df,
    batch_request=batch_request,
    expectation_suite_name=expectation_suite_name
)



In [0]:
validator.expect_column_values_to_not_be_null(column="customer_key")
validator.expect_column_values_to_be_unique(column="customer_key")
validator.expect_column_values_to_be_of_type(column="customer_key", type_="string")
validator.expect_column_values_to_not_be_null(column="name")
validator.expect_column_values_to_be_unique(column="name")
validator.expect_column_values_to_be_of_type(column="name", type_="string")

In [0]:
validator.save_expectation_suite(discarded_failed_expectations=False)

In [0]:
checkpoint_name = 'cust_dim_checkpoint'

In [0]:
checkpoint = Checkpoint(
    name = checkpoint_name,
    run_name_template = f"%Y%m%dT%H%M%S-{checkpoint_name}"
    data_context = context,
    batch_request = batch_request,
    expectation_suite_name = expectation_suite_name,
    action_list = [
        {
            "name":"store_validation_result",
            "action": {"class_name": "StoreValidationResultAction"}
        },
        {
            "name":"update_data_docs",
            "class_name": "UpdateDataDocsAction",
        },
    ],
    runtime_configuration = {
        "result_format": {"result_format": "SUMMARY"}
    }
)

In [0]:
context.add_or_update_checkpoint(checkpoint=checkpoint)

In [0]:
checkpoint_result = checkpoint.run()
print(checkpoint_result)